# 05 — Shared neural computation across models and sessions

**Question:** when two neural foundation-model architectures make the same prediction, do they rely on the same event-relative parts of an ORION representation?

This notebook follows the v0.3 chain: `RepresentationBatch → causal interventions → effect maps → cross-context stability → architecture comparison → falsifiable hypotheses`.

**Evidence level:** scientific synthetic. A hypothesis emitted here is a candidate interpretation, not a biological conclusion.

In [ ]:
import numpy as np
from orion.contracts import RepresentationBatch
from neuros_mechint.benchmarks import MechanismContext
from neuros_mechint.integrations.orion_study import (
    OrionRepresentationContext,
    run_shared_representation_study,
)

def make_context(context_id, architecture, session_id, origin_ns):
    timestamps = np.asarray([origin_ns - 10, origin_ns, origin_ns + 10, origin_ns + 20])
    values = np.asarray([[1.0, 0.1], [2.0, 0.2], [4.0, 0.4], [0.5, 0.05]])
    batch = RepresentationBatch(values=values, timestamps_ns=timestamps)
    def score(rep):
        return float(np.asarray(rep.values).sum())
    return OrionRepresentationContext(
        context=MechanismContext(
            context_id=context_id, architecture=architecture,
            dataset_id='synthetic-neural', session_id=session_id,
        ),
        representation=batch, scorer=score, alignment_origin_ns=origin_ns,
        alignment_label='task_event',
    )

contexts = [
    make_context('transformer-s1', 'transformer', 's1', 1_000),
    make_context('transformer-s2', 'transformer', 's2', 9_000),
    make_context('ssm-s1', 'ssm', 's1', 50_000),
    make_context('ssm-s2', 'ssm', 's2', 90_000),
]

In [ ]:
study = run_shared_representation_study(
    contexts, window_ns=10, stride_ns=10, top_k=2,
    include_feature_audits=True,
)
study.analysis.comparison.isolated_axis_stability['architecture'].to_dict()

In [ ]:
for hypothesis in study.analysis.hypotheses:
    print(hypothesis.hypothesis_id, hypothesis.priority)
    print(' ', hypothesis.statement)
    print('  falsify with:')
    for test in hypothesis.falsification_tests:
        print('   -', test)
print('scientific fingerprint:', study.study_fingerprint)
print('run provenance hash:', study.run_hash)

## What to do on real data

1. Align every context to the **same semantic event** (stimulus, movement onset, reward, etc.), not merely recording start.
2. Hold the downstream metric/task definition fixed and compare models at matched performance where possible.
3. Run multiple intervention families (zero, mean, donor patching, conditional resampling) and matched controls.
4. Use discovery sessions to generate hypotheses, then lock thresholds and test on held-out sessions/datasets.
5. Change one context axis at a time when possible so architecture, subject, session, and dataset effects are not confounded.
6. Require substantial shared intervention coverage; a high correlation on a tiny overlap is not strong mechanistic evidence.

A strong result is not ‘the maps look similar.’ It is that an aligned causal pattern survives controls, replication, alternative interventions, and held-out validation.